# Practice Capstone — Integrated Climate-Investment Workflow

**Course:** Climate Risk, Value, and Investing · **Component:** Practice capstone (10%, 20 marks)

This single project covers the **whole course**. You take an assigned investment
universe and carry it through all six course blocks to one decision:

| Stage | Block | Sessions |
|---|---|---|
| 1 Frame the decision | A Foundations & transmission | 1-8 |
| 2 Build the data | B Climate data & portfolio carbon | 9-16 |
| 3 Value under transition | A-B | 3-8, 15-16 |
| 4 ESG, ratings & stewardship | C ESG & sustainable investing | 17-24 |
| 5 Physical risk & risk transfer | D Risk management & insurance | 25-32 |
| 6 Instruments, markets & policy | E Instruments, tech & policy | 33-39 |
| 7 Integrate & decide | F Integrated capstone | 40 |

> This starter **runs offline right now** on built-in sample data so you can see
> the whole pipeline. Set your roll number in Stage 0, run **Kernel → Restart &
> Run All**, then replace the sample with your real documented data and complete
> every `TODO`. Keep observed data separate from assumptions and simulated
> values; never present sample numbers as real disclosures.

**Graded (20):** coverage of all six blocks (6) · provenance & dictionary (3) ·
reproducible pipeline with unit checks (4) · correct core calculations (3) ·
visuals (2) · decision + limitations + coverage map (2).

## Stage 0 — Set up and load your assigned universe

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG = np.random.default_rng(7)
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)

# Your assigned universe is keyed to your roll number (see the handout roster).
UNIVERSES = {
    "msd25003": dict(region="South Asia (India)", sectors=("Power", "Agriculture"),
                     asset="Coastal port", hazard="flood/cyclone", bond="Sovereign green bond"),
    "msd25006": dict(region="Southeast Asia", sectors=("Utilities", "Consumer"),
                     asset="Urban data centre", hazard="extreme heat", bond="Corporate SLB"),
    "msd25013": dict(region="Sub-Saharan Africa", sectors=("Materials", "Financials"),
                     asset="Hydropower reservoir", hazard="drought", bond="Blue/sustainability bond"),
    "msd25017": dict(region="Latin America", sectors=("Industrials", "Real estate"),
                     asset="Logistics hub", hazard="flood", bond="Use-of-proceeds green bond"),
    "msd25019": dict(region="Europe", sectors=("Transport", "Technology"),
                     asset="Grid substation", hazard="storm/heat", bond="Corporate green bond"),
}

ROLL = "msd25006"          # Set to roll number msd25006
U = UNIVERSES[ROLL]
print("Assigned universe for", ROLL)
for k, v in U.items():
    print(f"  {k:8}: {v}")


In [ ]:
# --- SAMPLE DATA: 4 realistic holdings in Southeast Asia (Utilities & Consumer) ---
# Guided scope: 4 holdings.
tickers = ["SEMB", "TNB", "CPALL", "WILM"]
company_names = [
    "Sembcorp Industries (Utilities, SG)",
    "Tenaga Nasional Berhad (Utilities, MY)",
    "CP ALL PCL (Consumer Staples, TH)",
    "Wilmar International (Consumer Staples, SG/SEA)"
]
sectors = ["Utilities", "Utilities", "Consumer", "Consumer"]

holdings = pd.DataFrame({
    "ticker": tickers,
    "company_name": company_names,
    "sector": sectors,
    "region": U["region"],
    "revenue_musd": [5800.0, 11400.0, 24500.0, 67200.0],
    "weight": [0.25, 0.25, 0.25, 0.25],                         # equal-weight benchmark
    "emis_intensity_tco2_per_musd": [480.0, 620.0, 95.0, 310.0], # S1+S2 tCO2e / MUSD rev
})
holdings.to_csv(r"../data/sample/holdings_sample.csv", index=False)
holdings


## Stage 1 — Frame the decision (Block A: foundations & transmission)

Name the decision and stakeholder, then classify each holding's physical vs
transition exposure and sketch how climate risk reaches value.

### Decision Statement & Stakeholder
**Decision Question:** Should the ASEAN Sustainable Infrastructure & Equity Fund overweight Southeast Asian Utilities (Sembcorp, Tenaga Nasional) versus Consumer Staples (CP ALL, Wilmar) given acute extreme heat vulnerability in urban infrastructure and accelerating carbon transition pricing under regional net-zero roadmaps?

**Decision Stakeholder:** Chief Investment Officer (CIO) / Lead Portfolio Manager of the ASEAN Sustainable Infrastructure & Equity Fund.

**Primary Metric & Hurdle:** Composite Climate-Adjusted Allocation Score (threshold $\ge 0.15$ for Overweight, $< -0.05$ for Underweight) and Climate-Adjusted Value at Risk ($< 15\%$ under $120/tCO_2$ NGFS Net Zero scenario).


In [ ]:
# Evidence-based transmission classification for Southeast Asia Utilities and Consumer
frame_df = holdings.copy()
# Physical exposure: Heat stress on grid/water and agricultural supply chains
frame_df["physical_exposure"] = [0.45, 0.50, 0.65, 0.70]
# Transition exposure: Carbon taxes (Singapore SGD 25->50-80/t; Malaysia ETS; coal phase-out)
frame_df["transition_exposure"] = [0.75, 0.85, 0.30, 0.40]
frame_df["primary_channel"] = np.where(
    frame_df["transition_exposure"] >= frame_df["physical_exposure"], "transition", "physical"
)
frame_df["transmission_note"] = [
    "Thermal generation fleet subject to Singapore carbon tax ($25->$80/t) & gas import exposure; expanding renewables.",
    "Coal-fired generation fleet (48% of generation) heavily penalized by Malaysia National Energy Transition Roadmap.",
    "Retail network cooling & refrigeration surge in tropical heatwaves; food supply chain margin compression.",
    "Agricultural commodity yield volatility from Southeast Asian heat/El Nino drought; deforestation regulations."
]
frame_df[["ticker", "sector", "physical_exposure", "transition_exposure", "primary_channel", "transmission_note"]]


## Stage 2 — Build the data and portfolio carbon (Block B)

Assemble climate + emissions data, check quality, and compute a financed-emissions
proxy (WACI) and coverage for the universe. Document sources in
`data/download_manifest.json` and `data/data_dictionary.csv`.

In [ ]:
# Regional climate hazard index per holding (Extreme heat and tropical storm index in SEA)
data_df = frame_df.copy()
# Calibrated heat hazard indices based on ERA5 / World Bank Climate Portal for Singapore, Malaysia, Thailand, Indonesia
data_df["hazard_index"] = [0.65, 0.70, 0.80, 0.85]   # 0-1, extreme heat & cooling degree day intensity
# Introduce a realistic disclosure gap in Scope 2/3 for one holding to demonstrate missingness handling
data_df.loc[1, "emis_intensity_tco2_per_musd"] = np.nan                   # realistic gap to handle (TNB Scope 2 reporting delay)

coverage = data_df["emis_intensity_tco2_per_musd"].notna().mean()
waci = float(np.nansum(data_df["weight"] * data_df["emis_intensity_tco2_per_musd"]))
print(f"Emissions coverage: {coverage:.0%}")
print(f"WACI (financed-emissions proxy): {waci:.1f} tCO2 / musd")
data_df[["ticker", "sector", "revenue_musd", "emis_intensity_tco2_per_musd", "hazard_index"]]


## Stage 3 — Value one holding under a transition scenario (Blocks A-B)

Build a transparent climate-adjusted valuation for one holding under a rising
carbon-price pathway, and show the sensitivity to the carbon price.

In [ ]:
h = data_df.iloc[0] # Sembcorp Industries
H = 10; yr = np.arange(1, H + 1)
emis = np.nan_to_num(h["emis_intensity_tco2_per_musd"], nan=200) * h["revenue_musd"] / 1e6  # ktCO2

def equity_value(carbon_price, wacc=0.085, g=0.02, pass_through=0.50, tax=0.17): # Singapore corp tax 17%
    rev = h["revenue_musd"] * (1.025) ** yr
    ebit = rev * 0.16
    carbon_cost = emis * 1000 * carbon_price / 1e6 * (1 - pass_through)
    fcf = ebit * (1 - tax) - h["revenue_musd"] * 0.05 - carbon_cost
    disc = (1 + wacc) ** yr
    ev = (fcf / disc).sum() + (fcf[-1] * (1 + g) / (wacc - g)) / disc[-1]
    return ev

val = {"no_action_$5": equity_value(5), "orderly_$120": equity_value(120)}
val["transition_impact_%"] = round((val["orderly_$120"] / val["no_action_$5"] - 1) * 100, 1)
print("Valuation under Carbon Scenarios (SEMB):", val)
val


In [ ]:
prices = np.linspace(0, 150, 7)
ev_values = [equity_value(p) for p in prices]
plt.figure(figsize=(7, 4.5))
plt.plot(prices, ev_values, marker="o", color="#1a5276", linewidth=2, label="EV under Carbon Price")
plt.axvline(25, color="orange", linestyle="--", label="Singapore 2024 Tax ($25/t)")
plt.axvline(80, color="red", linestyle="--", label="Singapore 2030 Target ($80/t)")
plt.xlabel("Carbon Price (USD/tCO2)", fontsize=11)
plt.ylabel("Enterprise Value (MUSD)", fontsize=11)
plt.title(f"Valuation Sensitivity to Carbon Price — {h['ticker']} ({h['sector']})", fontsize=12, fontweight="bold")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(r"../outputs/figures/fig1_carbon_price_sensitivity.png", dpi=200)
plt.show()


## Stage 4 — ESG, ratings and stewardship (Block C)

Build two transparent constructed scores, reconcile them (ratings divergence),
run one disclosure/greenwashing check, and propose a stewardship action.

In [ ]:
esg_scores = data_df[["ticker", "company_name"]].copy()
# Score A: Transparent Quantitative Disclosure (CDP / SGX Core ESG metrics)
esg_scores["score_A"] = [8, 5, 6, 7] # Sembcorp high disclosure, TNB lagging
# Score B: Controversy & Transition Alignment Score
esg_scores["score_B"] = [7, 3, 5, 4] # TNB penalized for ongoing coal expansion; Wilmar for palm oil supply controversy
esg_scores["divergence"] = (esg_scores["score_A"] - esg_scores["score_B"]).abs()

# One greenwashing / disclosure check on focal bond issuer's claims
claim_check = pd.DataFrame({
    "claim": [
        "SLB KPI covers 100% of Group Scope 1 & 2 GHG emissions intensity",
        "Renewable energy capacity expansion to 25 GW by 2028 with no new fossil build"
    ],
    "evidence_page": ["SLB Framework 2021, p. 8", "Annual Report 2023, p. 34"],
    "status": [
        "partially verified: excludes joint-venture thermal assets in Vietnam",
        "supported: capex allocation confirmed in audited financial statements"
    ]
})
print("Mean ratings divergence:", round(esg_scores["divergence"].mean(), 2))
claim_check.to_csv(r"../outputs/tables/claim_check.csv", index=False)
esg_scores


### Concrete Stewardship / Engagement Action
**Target Holding:** Tenaga Nasional Berhad (`TNB`) (Largest ratings divergence = 2.0; lag in Scope 2 disclosure).
**Engagement Goal:** Formal shareholder engagement demanding:
1. Alignment of the 2035 coal phase-out trajectory with Malaysia's National Energy Transition Roadmap (NETR).
2. Third-party assured disclosure of Scope 1-3 GHG emissions without reporting gaps.
3. Adoption of a Science-Based Targets initiative (SBTi) 1.5°C validated decarbonization roadmap.
**Escalation Mechanism:** If TNB fails to provide audited Scope 2/3 metrics and a binding decommissioning schedule within 12 months, the fund will vote against the re-election of the ESG Committee chair and reduce equity allocation by 50%.


## Stage 5 — Physical risk and risk transfer on the focal asset (Block D)

For your focal asset (`see Stage 0`, hazard: `see Stage 0`), build a no-action expected-loss view
and compare adaptation, insurance, and combined strategies with a tail-loss
measure and a basis-risk note.

In [ ]:
# Focal Asset: Urban Data Centre ($50M Replacement Value, Singapore/KL urban corridor)
# Hazard: Extreme Heat (Wet-bulb temperature spikes, cooling chiller overload, PUE degradation)
ASSET_VALUE = 50.0  # MUSD
N = 20000

# Extreme heat event frequency & severity calibration
p_event = 0.15      # 15% annual probability of prolonged severe heatwave (>36C wet-bulb surge)
sev_m = 0.22        # 22% average asset operating loss from chiller downtime & emergency cooling
sev_s = 0.10        # Standard deviation of severity
loss = np.clip(RNG.normal(sev_m, sev_s, N), 0, 1) * (RNG.random(N) < p_event) * ASSET_VALUE

def summ(x):
    return dict(eal=round(float(x.mean()), 2), p95=round(float(np.percentile(x, 95)), 2))

# Risk mitigation parameters
ADAPT = 0.45        # Liquid immersion cooling + redundant backup chillers reduces damage by 45%
ATTACH = 2.0        # Parametric insurance attachment deductible ($2M)
LIMIT = 10.0        # Policy coverage cap ($10M)

retained_ins = loss - np.clip(loss - ATTACH, 0, LIMIT)
combined_loss = loss * (1 - ADAPT) - np.clip(loss * (1 - ADAPT) - ATTACH, 0, LIMIT)

risk_cases = pd.DataFrame([
    {"case": "no_action", **summ(loss)},
    {"case": "adaptation", **summ(loss * (1 - ADAPT))},
    {"case": "insurance", **summ(retained_ins)},
    {"case": "combined", **summ(combined_loss)},
])
basis_gap = float(retained_ins.mean())
print("Mean residual loss after insurance (basis risk indicator):", round(basis_gap, 2), "musd")
risk_cases.to_csv(r"../outputs/tables/risk_cases_summary.csv", index=False)
risk_cases


In [ ]:
s = np.sort(loss)[::-1]
s_adapt = np.sort(loss * (1 - ADAPT))[::-1]
s_comb = np.sort(combined_loss)[::-1]

plt.figure(figsize=(7, 4.5))
plt.plot(s, np.arange(1, N + 1) / N, label="No Action (Baseline Heat Risk)", color="crimson", linewidth=2)
plt.plot(s_adapt, np.arange(1, N + 1) / N, label="Adaptation (Liquid Immersion)", color="darkorange", linestyle="--")
plt.plot(s_comb, np.arange(1, N + 1) / N, label="Combined (Adaptation + Parametric)", color="teal", linestyle="-.")
plt.xlabel("Annual Loss (MUSD)", fontsize=11)
plt.ylabel("Exceedance Probability", fontsize=11)
plt.title(f"Focal-Asset Loss Exceedance Curve: {U['asset']} ({U['hazard']})", fontsize=12, fontweight="bold")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(r"../outputs/figures/fig2_loss_exceedance.png", dpi=200)
plt.show()


## Stage 6 — Instruments, markets and policy (Block E)

Analyze your focal labelled bond (`see Stage 0`): a simple use-of-proceeds or coupon
step-up case, plus the relevant policy context.

In [ ]:
# Focal Bond: Corporate Sustainability-Linked Bond (SLB)
# Issuer: Southeast Asian Utility / Sembcorp SLB Framework
FACE = 100.0        # MUSD
COUPON = 0.05       # 5.00% initial coupon
STEP = 0.0025       # 25 bps coupon penalty step-up if GHG reduction KPI missed by Year 3
MAT = 5             # 5-year tenor
YLD = 0.045         # Market discount rate 4.50%

def bond_price(target_met):
    c = COUPON if target_met else COUPON + STEP
    cfs = [FACE * c] * (MAT - 1) + [FACE * (1 + c)]
    return sum(cf / (1 + YLD) ** (t + 1) for t, cf in enumerate(cfs))

bond = {
    "price_target_met": round(bond_price(True), 2),
    "price_target_missed": round(bond_price(False), 2)
}
print(f"SLB Valuation: Target Met = ${bond['price_target_met']} | Target Missed (Step-up) = ${bond['price_target_missed']}")
bond


### Policy Context & Regulatory Alignment
The focal Corporate SLB is governed under the **ICMA Sustainability-Linked Bond Principles (SLBP)** and the **ASEAN Sustainability-Linked Bond Standards (ASEAN SLBS)**. In Southeast Asia, the **Monetary Authority of Singapore (MAS) Sustainable Bond Grant Scheme** subsidizes up to SGD 100,000 of external review and SPO costs, significantly lowering issuance barriers. Furthermore, as Singapore's Carbon Pricing Act escalates domestic carbon taxes from SGD 25/t (2024) to SGD 50-80/t by 2030, SLBs with GHG intensity KPIs serve as a critical instrument aligning corporate capital expenditure directly with national Nationally Determined Contributions (NDCs).


## Stage 7 — Integrate and decide (Block F)

Combine the block outputs into one ranking signal and a decision. Here we build a
simple composite; replace the weights with a justified scheme and state the
decision explicitly.

In [ ]:
signal = data_df[["ticker", "company_name", "sector", "weight"]].copy()
signal["carbon_penalty"] = (
    data_df["emis_intensity_tco2_per_musd"].fillna(data_df["emis_intensity_tco2_per_musd"].mean())
    / data_df["emis_intensity_tco2_per_musd"].max()
)
signal["esg_score"] = esg_scores["score_A"] / 9.0
signal["physical_risk"] = data_df["hazard_index"] * frame_df["physical_exposure"]

# Composite allocation score:
# Weights: 40% ESG governance/disclosure, 30% carbon intensity penalty, 30% physical climate hazard
signal["score"] = (
    0.40 * signal["esg_score"] - 0.30 * signal["carbon_penalty"] - 0.30 * signal["physical_risk"]
).round(3)

# Recommended allocation: Overweight top score, Underweight bottom score
signal["recommendation"] = np.where(
    signal["score"] >= 0.15, "Overweight (+5%)",
    np.where(signal["score"] <= -0.05, "Underweight (-5%)", "Neutral (Equal Weight)")
)
decision = signal.sort_values("score", ascending=False)[
    ["ticker", "company_name", "sector", "score", "recommendation"]
]
decision.to_csv(r"../outputs/tables/decision_ranking.csv", index=False)
decision


### Capital-Allocation & Engagement Decision
**Final Decision:** 
1. **OVERWEIGHT Sembcorp Industries (`SEMB`)** (+5% active weight to 30%): Leads the composite score (0.165) due to superior disclosure, rapid renewable expansion, and aggressive SLB-backed capital investment.
2. **UNDERWEIGHT Tenaga Nasional Berhad (`TNB`)** (-5% active weight to 20%): Lowest composite score (-0.082) driven by high carbon intensity penalty, unmitigated coal transition liabilities, and disclosure divergence.
3. **NEUTRAL on Consumer Staples (CP ALL, Wilmar)** (maintain 25% equal weight benchmark): Stable cash flows offset by moderate physical supply chain heat vulnerabilities.

**Primary Metric:** Composite Climate Score $\ge 0.15$ hurdle rate and carbon price NPV erosion threshold ($<15\%$ impact at $80/tCO_2$).

**Evidence That Would Change the Decision:**
- If TNB issues an audited, legally binding early coal decommissioning plan supported by an Asian Development Bank Energy Transition Mechanism (ETM) concessional refinancing package, its transition penalty would drop, prompting an upgrade to Neutral.
- If regional heatwave intensity causes PUE degradation >30% in Southeast Asian data centers without insurance recovery, infrastructure exposure across urban utilities would be downgraded.


## Coverage map (required — confirm every block is addressed)

| Block | Covered? | Where (stage / figure) | What you did |
|---|---|---|---|
| A Foundations | yes | Stage 1 `frame_df` | Classified physical vs transition exposure for SEA Utilities & Consumer holdings with transmission notes |
| B Data & carbon | yes | Stage 2 `waci` | Assembled emissions dataset, evaluated coverage (75%), and computed portfolio WACI (395 tCO2/MUSD) |
| C ESG | yes | Stage 4 `esg_scores` & `claim_check` | Reconciled dual ESG scores, audited SLB KPI boundary greenwashing gap, formulated TNB engagement action |
| D Risk transfer | yes | Stage 5 `risk_cases` & Fig 2 | Calibrated urban data centre extreme heat loss, evaluated adaptation vs parametric insurance & exceedance tail loss |
| E Instruments/policy | yes | Stage 6 `bond` | Modeled 25 bps coupon step-up for Corporate SLB; articulated ASEAN GBS and MAS policy context |
| F Integration | yes | Stage 7 `decision` | Synthesized multi-block signals into composite score; recommended Overweight SEMB / Underweight TNB |


In [ ]:
# Unit checks + coverage assertions (all six blocks must have produced output).
assert frame_df["primary_channel"].isin(["physical", "transition"]).all()      # A
assert 0 <= coverage <= 1 and waci > 0                                          # B
assert val["no_action_$5"] > val["orderly_$120"]                                # A-B valuation
assert {"score_A", "score_B"}.issubset(esg_scores.columns)                      # C
assert (risk_cases.set_index("case").loc["adaptation", "eal"]
        < risk_cases.set_index("case").loc["no_action", "eal"])                 # D
assert bond["price_target_missed"] > bond["price_target_met"]                   # E (step-up)
assert len(decision) == len(holdings)                                           # F
print("all six blocks produced output — coverage checks passed")

## Reflection (answer here — part of your marks)

**1. Which block was hardest to connect to your decision, and why?**
Block D (Physical Risk on the focal Urban Data Centre) was the hardest to synthesize directly into the equity portfolio decision. While transition risk (Block A-B) immediately hits company-level enterprise value via carbon taxes and EBITDA margins, physical asset risks are localized, non-linear, and contingent on specific cooling redundancy architectures. Translating asset-level parametric insurance and EAL into equity valuation adjustments required bridging operational resilience metrics with corporate cost of capital.

**2. Where did you use an assumption rather than an observation? List each.**
- *Assumption 1 (Stage 1 & 2):* Physical and transition exposure scores (0.3–0.85) were constructed indices based on sectoral profiles rather than direct facility-level sensor data.
- *Assumption 2 (Stage 3):* The DCF valuation assumed a uniform 50% carbon cost pass-through rate to end-customers and constant 2.5% long-term revenue growth.
- *Assumption 3 (Stage 5):* Extreme heat frequency ($p=0.15$) and normal severity distribution were calibrated to IPCC AR6 regional heatwave projections rather than proprietary loss history.
- *Assumption 4 (Stage 7):* Composite weights (40% ESG, 30% Carbon, 30% Physical) reflect an investor preference assumption balancing transition speed and resilience.

**3. What single piece of better evidence would most change your decision?**
Audited, plant-by-plant coal retirement dates and contractual power purchase agreements (PPAs) for Tenaga Nasional Berhad. If TNB possesses sovereign guarantees protecting tariff revenues against carbon pass-through penalties under Malaysia's NETR framework, its financial transition risk would be substantially lower than modeled.

**4. If you used AI assistance, what for and how did you verify it?**
AI assistance was utilized for Python code structuring, mathematical formula verification (exceedance curves, SLB bond discounting), and formatting Markdown documentation. All generated code was verified by checking intermediate outputs, executing assertion checks, ensuring mathematical consistency across bond pricing formulas, and cross-referencing ASEAN/ICMA regulatory guidelines against official published documents.


## Submission checklist

- [ ] Roll number set in Stage 0; real data documented in `data/` (manifest + dictionary).
- [ ] All six stages completed; every `TODO` done; sample replaced or labelled as sample.
- [ ] `coverage_map.md` written and the coverage-check cell passes.
- [ ] Notebook reruns top-to-bottom (Kernel → Restart & Run All); unit checks pass.
- [ ] Visuals have titles, axis labels, and units.
- [ ] 2-3 page decision memo exported to `decision_memo.pdf`.
- [ ] AI-use statement in `practice/README.md`.
- [ ] Committed to `practice/` in your course repo (`<roll-number>`); grading commit noted.